# Learning from quantum data
**Quantum Machine Learning · Week 4 (QML for research) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/qml/w4d3.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

**Learning from quantum data.** When the data are quantum states, the learner decides how to measure them, and a
bad choice destroys the information for good. The best anyone can do is set by the Helstrom bound; a trainable
measurement can learn to reach it, while a learner that only sees fixed measurement records can be stuck at a coin flip.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import DensityMatrix

def state(theta, phi=0.0, noise=0.0):
    """The qubit RZ(φ)·RY(θ)|0⟩, mixed with a fraction `noise` of white noise, as a 2×2 density matrix."""
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    qc.rz(phi, 0)
    return (1 - noise) * DensityMatrix(qc).data + noise * np.eye(2) / 2

def helstrom(rho0, rho1):
    """The best possible chance of telling rho0 from rho1 from one copy, with equal odds."""
    return 0.5 + 0.25 * np.sum(np.abs(np.linalg.eigvalsh(rho0 - rho1)))

for name, r0, r1 in [("|0⟩ versus |1⟩", state(0), state(np.pi)),
                     ("30° apart", state(0), state(np.pi / 6)),
                     ("|+⟩ versus |−⟩", state(np.pi / 2), state(np.pi / 2, np.pi)),
                     ("|+⟩ versus |−⟩, 40% noise", state(np.pi / 2, 0, 0.4), state(np.pi / 2, np.pi, 0.4))]:
    print(f"{name:28s} best possible success: {helstrom(r0, r1):.3f}")

## A quantum dataset
A sensor hands us one qubit per event. Both classes have the same tilt, so they look identical in the Z basis;
the class is hidden in the phase: around 0.6 rad for class 0, around 0.6 + π for class 1.

In [ ]:
def sensor(n, seed):
    rng = np.random.default_rng(seed)
    labels = rng.integers(0, 2, n)
    phis = 0.6 + np.pi * labels + 0.25 * rng.normal(size=n)
    return [state(1.2, phi) for phi in phis], labels

train_states, y_train = sensor(200, 0)
test_states, y_test = sensor(400, 1)
avg0 = np.mean([s for s, c in zip(train_states, y_train) if c == 0], axis=0)
avg1 = np.mean([s for s, c in zip(train_states, y_train) if c == 1], axis=0)
Z = np.diag([1.0, -1.0])
print("chance of reading 0 in the Z basis, class 0 and class 1:",
      round(np.real(np.trace(avg0 @ (np.eye(2) + Z) / 2)), 3), round(np.real(np.trace(avg1 @ (np.eye(2) + Z) / 2)), 3))
print(f"best possible success on this data (Helstrom): {helstrom(avg0, avg1):.3f}")

## A trainable measurement
Rotate each state with $R_Z(\beta)$ then $R_Y(\alpha)$, measure Z, and call outcome 0 "class 0". The chance of
being right is an expectation value, so the parameter-shift rule gives its gradient.

In [ ]:
def rz(t):
    return np.diag([np.exp(-1j * t / 2), np.exp(1j * t / 2)])

def ry(t):
    c, s = np.cos(t / 2), np.sin(t / 2)
    return np.array([[c, -s], [s, c]])

def p_zero(rho, alpha, beta):
    """Chance that the rotated state reads 0."""
    U = ry(alpha) @ rz(beta)
    return np.real((U @ rho @ U.conj().T)[0, 0])

def success(states, labels, alpha, beta):
    p0 = np.array([p_zero(s, alpha, beta) for s in states])
    return np.mean(np.where(labels == 0, p0, 1 - p0))

alpha, beta = 0.3, 0.2                                         # start somewhere arbitrary
for step in range(60):
    shift = np.pi / 2
    g_a = (success(train_states, y_train, alpha + shift, beta) - success(train_states, y_train, alpha - shift, beta)) / 2
    g_b = (success(train_states, y_train, alpha, beta + shift) - success(train_states, y_train, alpha, beta - shift)) / 2
    alpha, beta = alpha + 0.8 * g_a, beta + 0.8 * g_b          # gradient ascent: maximise the success
    if step % 20 == 0:
        print(f"step {step:2d}: training success {success(train_states, y_train, alpha, beta):.3f}")
print(f"learned measurement: α = {alpha:.2f}, β = {beta:.2f}")

rng = np.random.default_rng(2)
one_shot = np.array([rng.random() < p_zero(s, alpha, beta) for s in test_states])   # True means it read 0
print(f"expected test success (exact probabilities): {success(test_states, y_test, alpha, beta):.3f}")
print(f"one round of ONE measurement per test state: {np.mean((~one_shot).astype(int) == y_test):.3f}   (wobbles by about ±0.01)")
test0 = np.mean([s for s, c in zip(test_states, y_test) if c == 0], axis=0)
test1 = np.mean([s for s, c in zip(test_states, y_test) if c == 1], axis=0)
print(f"best possible on the test states (Helstrom): {helstrom(test0, test1):.3f}")

## A learner that only gets classical records
Measure every state in the Z basis first, then hand the bits to the best classical learner there is.

In [ ]:
bits_train = np.array([rng.random() < p_zero(s, 0.0, 0.0) for s in train_states])
bits_test = np.array([rng.random() < p_zero(s, 0.0, 0.0) for s in test_states])
for bit in (True, False):                                      # the best classical rule: majority class for each bit value
    sel = bits_train == bit
    print(f"read {'0' if bit else '1'}: {sel.sum()} training states, {np.mean(y_train[sel]):.2f} of them class 1")
rule = {bit: int(np.mean(y_train[bits_train == bit]) > 0.5) for bit in (True, False)}
print(f"test success of the best rule on Z records: {np.mean(np.array([rule[b] for b in bits_test]) == y_test):.3f}")

## Your turn: a noisy sensor
Mix every state with white noise of strength $p$. For $p = 0, 0.2, 0.4, 0.6$, compute the Helstrom bound, retrain
the measurement, and measure its test success with one shot per state.

In [ ]:
def noisy(states, p):
    return [(1 - p) * s + p * np.eye(2) / 2 for s in states]

# TODO: for p in (0, 0.2, 0.4, 0.6): bound, training (reuse the loop above), and one-shot test success

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def noisy(states, p):
    return [(1 - p) * s + p * np.eye(2) / 2 for s in states]

def train_measurement(states, labels, steps=60):
    a, b = 0.3, 0.2
    for _ in range(steps):
        g_a = (success(states, labels, a + np.pi / 2, b) - success(states, labels, a - np.pi / 2, b)) / 2
        g_b = (success(states, labels, a, b + np.pi / 2) - success(states, labels, a, b - np.pi / 2)) / 2
        a, b = a + 0.8 * g_a, b + 0.8 * g_b
    return a, b

for p in (0, 0.2, 0.4, 0.6):
    tr, te = noisy(train_states, p), noisy(test_states, p)
    a, b = train_measurement(tr, y_train)
    t0 = np.mean([s for s, c in zip(te, y_test) if c == 0], axis=0)
    t1 = np.mean([s for s, c in zip(te, y_test) if c == 1], axis=0)
    print(f"p = {p:.1f}: best possible {helstrom(t0, t1):.3f}, learned measurement {success(te, y_test, a, b):.3f} (expected)")
print("White noise shrinks the gap between the two classes' states, and the bound falls with it: past the bound,")
print("no learner, quantum or classical, can do better from one copy.")